# 15.3 矩阵分解 / Matrix Factorization

**中文**：上一节的邻域 CF 有两个硬伤：稀疏（很多用户对没有共同评分）和不可扩展（$O(n^2)$ 相似度矩阵）。**矩阵分解（Matrix Factorization, MF）** 一举解决两者，是 **Netflix Prize 夺冠方案的核心**，也是直到今天所有"embedding 推荐"的鼻祖。
**English**: Neighborhood CF has two weaknesses: sparsity (many user pairs share no items) and non-scalability ($O(n^2)$ similarity matrix). **Matrix Factorization (MF)** solves both at once — the **core of the Netflix Prize-winning solution** and the ancestor of every "embedding-based" recommender to this day.

---

**中文**：核心思想：把巨大、稀疏的评分矩阵 $R$（用户×物品）**近似分解**成两个又瘦又长的矩阵相乘：
**English**: Core idea: approximately **factor** the huge, sparse rating matrix $R$ (users × items) into a product of two tall-thin matrices:

$$R \approx P Q^\top,\qquad \hat r_{ui} = \mathbf{p}_u \cdot \mathbf{q}_i = \sum_{f=1}^{k} p_{uf}\,q_{if}$$

**中文**：$\mathbf{p}_u \in \mathbb{R}^k$ 是用户 $u$ 的 **$k$ 维隐向量（latent factors）**，$\mathbf{q}_i \in \mathbb{R}^k$ 是物品 $i$ 的隐向量。$k$ 通常只有 20~200，远小于物品数。预测评分就是两个隐向量的点积。
**English**: $\mathbf{p}_u \in \mathbb{R}^k$ is user $u$'s **$k$-dim latent factor vector**, $\mathbf{q}_i \in \mathbb{R}^k$ is item $i$'s. $k$ is typically only 20–200, far smaller than #items. The predicted rating is the dot product of the two latent vectors.

**中文**：直觉：每一维隐因子可以理解为一个"潜在主题"，比如"是否文艺片""是否动作多""是否适合儿童"。$p_{uf}$ 是用户对该主题的偏好强度，$q_{if}$ 是物品在该主题上的含量。**这些主题不是人工定义的，而是从数据里自动学出来的。**
**English**: Intuition: each latent dimension is a learned "theme" — e.g. "artsy-ness," "action-ness," "kid-friendliness." $p_{uf}$ is the user's preference strength on that theme, $q_{if}$ is the item's loading on it. **These themes are not hand-defined — they are learned automatically from data.**

> 💡 **面试速查 / Interview cheat-sheet（★★★ 必考）**
> **中文**：MF 把"用户/物品"都嵌入到同一个低维空间，用点积打分。优点：① 隐式地解决稀疏（即使两人没共同评分，也能通过隐空间关联）；② 可扩展（参数量 $O((m+n)k)$ 而非 $O(n^2)$）；③ 是后续所有深度推荐（双塔、DeepFM）的基石。关键组件：**偏置项**（全局/用户/物品）、**正则化**、训练用 **SGD 或 ALS**。**为什么不直接用 SVD？** 因为经典 SVD 要求矩阵无缺失，而评分矩阵 95% 缺失，用 0 填充会把"没评分"误当成"不喜欢"，引入巨大偏差。
> **English**: MF embeds users and items into one low-dim space and scores by dot product. Pros: ① implicitly handles sparsity (two users with no shared items can still relate via latent space); ② scalable ($O((m+n)k)$ params, not $O(n^2)$); ③ the foundation of all later deep recommenders (two-tower, DeepFM). Key parts: **bias terms** (global/user/item), **regularization**, trained by **SGD or ALS**. **Why not plain SVD?** Classic SVD needs a complete matrix; the rating matrix is 95% missing, and filling with 0 wrongly treats "unrated" as "disliked," causing huge bias.


In [ ]:

# ============================================================
# 数据 + 演示"为什么朴素 SVD 不行" / Data + why naive SVD fails
# ============================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(0)
R_DIR=os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")
ratings=pd.read_csv(os.path.join(R_DIR,"u.data"),sep="\t",names=["user","item","rating","ts"])
rs=ratings.sort_values("ts"); tr=[];te=[]
for _,g in rs.groupby("user"):
    c=int(len(g)*0.8); tr.append(g.iloc[:c]); te.append(g.iloc[c:])
train=pd.concat(tr); test=pd.concat(te)

# 重新编号成连续 0..m-1 / 0..n-1（隐向量按下标存）/ reindex to contiguous ids
uids=np.sort(ratings["user"].unique()); iids=np.sort(ratings["item"].unique())
u2x={u:x for x,u in enumerate(uids)}; i2x={i:x for x,i in enumerate(iids)}
m,n=len(uids),len(iids)
def to_arrays(df):
    return (df["user"].map(u2x).values, df["item"].map(i2x).values, df["rating"].values.astype(float))
ur,ir,rr = to_arrays(train)          # 训练三元组 (user_idx, item_idx, rating)
ur_te,ir_te,rr_te = to_arrays(test)  # 测试三元组
mu = rr.mean()                       # 全局平均分 / global mean

# --- 朴素 SVD：把缺失填 0 再做截断 SVD --- naive: fill missing with 0, truncated SVD
Rfull=np.zeros((m,n))
for x,y,r in zip(ur,ir,rr): Rfull[x,y]=r
U,S,Vt=np.linalg.svd(Rfull, full_matrices=False)   # 完整 SVD / full SVD
k=20
approx = (U[:,:k]*S[:k]) @ Vt[:k]                   # 取前k个奇异值重构 / rank-k reconstruction
pred_svd = approx[ur_te, ir_te]                     # 预测测试评分 / predict test ratings
rmse_svd = np.sqrt(np.mean((pred_svd-rr_te)**2))
print(f"训练三元组 / train triples: {len(rr)}, 用户 m={m}, 物品 n={n}")
print(f"朴素 SVD(填0) RMSE / naive zero-filled SVD: {rmse_svd:.4f}  (越大越糟 / worse is larger)")


**中文**：朴素 SVD 的 RMSE 会大得离谱（远大于上一节的 ~1.0）——因为它努力去拟合那些被填成 0 的"缺失"格子，把"没看过"硬学成"评 0 分"。正确做法是：**只在观测到的评分上**定义损失，让模型对缺失格子保持沉默。这就是 **FunkSVD**（Simon Funk 在 Netflix Prize 中提出）的精髓。
**English**: Naive SVD yields an absurdly large RMSE (far worse than the ~1.0 from the last section) — it wastes capacity fitting the zero-filled "missing" cells, learning "unrated" as "rated 0." The fix: define the loss **only over observed ratings**, staying silent on missing cells. This is the essence of **FunkSVD** (Simon Funk, Netflix Prize).

$$\min_{P,Q,b}\;\sum_{(u,i)\in\mathcal{K}}\Big(r_{ui}-\mu-b_u-b_i-\mathbf{p}_u\!\cdot\!\mathbf{q}_i\Big)^2 + \lambda\big(\|\mathbf{p}_u\|^2+\|\mathbf{q}_i\|^2+b_u^2+b_i^2\big)$$

**中文**：逐项解释：$\mathcal{K}$ 是**所有已观测评分**的集合（关键！只对它们求和）；$\mu$ 全局均值；$b_u,b_i$ 用户/物品**偏置**（有人天生打分高、有的电影天生评分高）；$\mathbf{p}_u\!\cdot\!\mathbf{q}_i$ 是个性化交互；$\lambda$ 是 L2 正则强度，防止隐向量过拟合稀疏数据。
**English**: Term by term: $\mathcal{K}$ is the set of **observed ratings only** (crucial — sum over these alone); $\mu$ global mean; $b_u,b_i$ user/item **biases** (some users rate high, some movies are universally rated high); $\mathbf{p}_u\!\cdot\!\mathbf{q}_i$ the personalized interaction; $\lambda$ the L2 regularization that prevents the latent vectors from overfitting sparse data.


In [ ]:

# ============================================================
# FunkSVD via SGD：只在观测评分上做随机梯度下降 / SGD over observed ratings only
# ============================================================
def train_funksvd(k=20, lr=0.01, reg=0.05, epochs=20, verbose=True):
    rng=np.random.default_rng(0)
    P=rng.normal(0,0.1,(m,k))      # 用户隐矩阵 / user factors (m,k)
    Q=rng.normal(0,0.1,(n,k))      # 物品隐矩阵 / item factors (n,k)
    bu=np.zeros(m); bi=np.zeros(n) # 用户/物品偏置 / biases
    idx=np.arange(len(rr)); hist=[]
    for ep in range(epochs):
        rng.shuffle(idx)                                # 每轮打乱样本顺序 / shuffle
        for t in idx:
            u,i,r=ur[t],ir[t],rr[t]
            pred=mu+bu[u]+bi[i]+P[u]@Q[i]               # 当前预测 / current prediction
            e=r-pred                                     # 残差 / residual
            # 梯度下降更新（每个参数 = 旧值 + lr*(误差梯度 - 正则)）
            bu[u]+=lr*(e-reg*bu[u]); bi[i]+=lr*(e-reg*bi[i])
            Pu=P[u].copy()
            P[u]+=lr*(e*Q[i]-reg*P[u])                  # 更新用户向量 / update p_u
            Q[i]+=lr*(e*Pu  -reg*Q[i])                  # 更新物品向量（用旧 p_u）/ update q_i
        # 每轮记录测试 RMSE / test RMSE each epoch
        pr=mu+bu[ur_te]+bi[ir_te]+np.sum(P[ur_te]*Q[ir_te],axis=1)
        rm=np.sqrt(np.mean((pr-rr_te)**2)); hist.append(rm)
        if verbose and (ep%4==0 or ep==epochs-1): print(f"  epoch {ep:2d}  test RMSE {rm:.4f}")
    return P,Q,bu,bi,hist

print("训练 FunkSVD (k=20) / training FunkSVD:")
P,Q,bu,bi,hist = train_funksvd(k=20, epochs=20)
rmse_funk=hist[-1]
print(f"FunkSVD 最终 RMSE / final: {rmse_funk:.4f}  vs 朴素SVD {rmse_svd:.2f}")


**中文**：FunkSVD 用 SGD 一条评分一条评分地更新。另一种主流训练法是 **ALS（交替最小二乘）**：固定 $Q$ 时，每个 $\mathbf{p}_u$ 有**闭式最优解**（一个岭回归）；再固定 $P$ 解每个 $\mathbf{q}_i$，交替进行。ALS 的优势是**每步都是凸的、可并行**（每个用户独立求解），所以是 Spark MLlib 等大数据框架的默认实现，尤其适合隐式反馈（下一节）。
**English**: FunkSVD updates one rating at a time via SGD. The other mainstream trainer is **ALS (Alternating Least Squares)**: with $Q$ fixed, each $\mathbf{p}_u$ has a **closed-form optimum** (a ridge regression); then fix $P$ and solve each $\mathbf{q}_i$, alternating. ALS is **convex per step and parallelizable** (each user solved independently) — the default in Spark MLlib, especially for implicit feedback (next section).

$$\mathbf{p}_u = \Big(Q_u^\top Q_u + \lambda I\Big)^{-1} Q_u^\top \mathbf{r}_u$$

**中文**：其中 $Q_u$ 是用户 $u$ 评过的物品的隐向量堆成的矩阵，$\mathbf{r}_u$ 是对应的（去偏后）评分。这就是一个标准的岭回归正规方程。
**English**: where $Q_u$ stacks the latent vectors of items $u$ rated and $\mathbf{r}_u$ the corresponding (de-biased) ratings — a standard ridge-regression normal equation.


In [ ]:

# ============================================================
# ALS：交替最小二乘（这里用全局均值去偏，不带 per-user/item 偏置以保持闭式简洁）
# ALS: alternating ridge regressions (de-bias with global mean for closed-form simplicity)
# ============================================================
# 预存每个用户评过的(item, rating)与每个物品被评过的(user, rating)/ build adjacency lists
from collections import defaultdict
by_user=defaultdict(list); by_item=defaultdict(list)
for u,i,r in zip(ur,ir,rr):
    by_user[u].append((i,r-mu)); by_item[i].append((u,r-mu))   # 评分减去全局均值 / de-bias

def train_als(k=20, reg=0.1, epochs=12):
    # 用 weighted-λ 正则(ALS-WR, Zhou 2008 Netflix 论文)：正则强度 ∝ 该行的评分数，
    # 这样评分少的冷用户/冷物品被更强地拉向 0，避免在稀疏行上过拟合。
    # Weighted-λ regularization (ALS-WR, Zhou 2008): λ scaled by #ratings of the row,
    # so cold users/items (few ratings) are pulled harder toward 0, preventing overfit.
    rng=np.random.default_rng(1)
    P=rng.normal(0,0.1,(m,k)); Q=rng.normal(0,0.1,(n,k))
    Ik=np.eye(k); hist=[]
    for ep in range(epochs):
        # --- 固定 Q，解所有 p_u ---
        for u in range(m):
            if not by_user[u]: continue
            its=np.array([i for i,_ in by_user[u]]); rs_=np.array([r for _,r in by_user[u]])
            Qu=Q[its]                                  # 该用户评过物品的隐向量 (d,k)
            P[u]=np.linalg.solve(Qu.T@Qu+reg*len(its)*Ik, Qu.T@rs_)  # 加权岭回归闭式解 / weighted ridge
        # --- 固定 P，解所有 q_i ---
        for i in range(n):
            if not by_item[i]: continue
            us=np.array([u for u,_ in by_item[i]]); rs_=np.array([r for _,r in by_item[i]])
            Pi=P[us]
            Q[i]=np.linalg.solve(Pi.T@Pi+reg*len(us)*Ik, Pi.T@rs_)
        pr=mu+np.sum(P[ur_te]*Q[ir_te],axis=1)
        rm=np.sqrt(np.mean((pr-rr_te)**2)); hist.append(rm)
    return P,Q,hist

print("训练 ALS-WR (k=20) / training ALS-WR:")
Pa,Qa,hist_als = train_als(k=20, reg=0.1, epochs=12)
print(f"ALS 最终 RMSE / final: {hist_als[-1]:.4f}")
print(f"\n{'方法/method':<26}{'RMSE':>8}")
for name,v in [("朴素SVD naive SVD",rmse_svd),("用户均值 user-mean",np.sqrt(np.mean((mu+0*rr_te-rr_te)**2))),
               ("FunkSVD (SGD)",rmse_funk),("ALS",hist_als[-1])]:
    print(f"{name:<26}{v:>8.4f}")


**中文**：现在做 MF 最迷人的部分——**看看隐空间里到底学到了什么**。我们把每部电影的 $k$ 维隐向量用 PCA 压到 2 维画出来，并按电影的"主类型"上色。如果 MF 真的学到了语义，**同类型的电影会自然聚到一起**——而我们从没把类型标签喂给模型，它纯粹从评分行为里学出了这种结构。
**English**: Now the most fascinating part of MF — **seeing what the latent space learned**. We PCA-project each movie's $k$-dim latent vector to 2D and color by its primary genre. If MF truly learned semantics, **same-genre movies cluster together** — even though we never fed it any genre labels; it discovered this structure purely from rating behavior.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(15,4.3))

# ① 训练曲线：FunkSVD vs ALS / training curves
ax[0].plot(range(1,len(hist)+1),hist,"o-",label="FunkSVD (SGD)",color="#4C72B0")
ax[0].plot(range(1,len(hist_als)+1),hist_als,"s-",label="ALS",color="#55A868")
ax[0].axhline(rmse_funk,ls=":",color="gray")
ax[0].set_title("收敛曲线 / convergence"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("test RMSE"); ax[0].legend()

# ② 隐因子个数 k 对 RMSE 的影响 / effect of #factors
ks=[2,5,10,20,40,80]
rmk=[train_funksvd(k=kk,epochs=15,verbose=False)[4][-1] for kk in ks]
ax[1].plot(ks,rmk,"o-",color="#C44E52")
ax[1].set_title("隐因子数 k 对 RMSE / #factors vs RMSE"); ax[1].set_xlabel("k (latent dim)"); ax[1].set_ylabel("test RMSE")

# ③ 物品隐空间 PCA + 按主类型上色 / item latent space colored by primary genre
GEN=["unknown","Action","Adventure","Animation","Children","Comedy","Crime","Documentary",
     "Drama","Fantasy","FilmNoir","Horror","Musical","Mystery","Romance","SciFi","Thriller","War","Western"]
cols=["item","title","d","v","url"]+GEN
mv=pd.read_csv(os.path.join(R_DIR,"u.item"),sep="|",encoding="latin-1",header=None,names=cols).set_index("item")
# 每部电影主类型 = 它拥有的类型里 idf 最高(最有区分度)的一个 / primary = rarest owned genre
Gmat=mv[GEN].values.astype(float); idf=np.log(len(mv)/(1+Gmat.sum(0)))
focus=["Animation","Horror","FilmNoir","Documentary","Western"]   # 选几个区分度高的类型 / distinctive genres
fcol={"Animation":"#1f77b4","Horror":"#d62728","FilmNoir":"#2ca02c","Documentary":"#9467bd","Western":"#ff7f0e"}
# PCA 把 Q 降到 2 维 / PCA Q -> 2D
Qc=Q-Q.mean(0); _,_,Vt2=np.linalg.svd(Qc,full_matrices=False); emb=Qc@Vt2[:2].T
for g in focus:
    pts=[]
    for i,iid in enumerate(iids):
        if iid in mv.index and mv.loc[iid,g]==1:
            owned=[gg for gg in focus if mv.loc[iid,gg]==1]
            if owned and max(owned,key=lambda x:idf[GEN.index(x)])==g:   # 主类型是 g / primary is g
                pts.append(emb[i])
    if pts:
        pts=np.array(pts); ax[2].scatter(pts[:,0],pts[:,1],s=14,alpha=0.7,c=fcol[g],label=f"{g}({len(pts)})")
ax[2].set_title("物品隐空间(从评分学出) / item latent space"); ax[2].legend(fontsize=8); ax[2].set_xlabel("PC1"); ax[2].set_ylabel("PC2")
plt.tight_layout(); plt.savefig("/tmp/rec03_viz.png",dpi=80); plt.show()
print("最佳 k / best #factors:", ks[int(np.argmin(rmk))], " RMSE=",round(min(rmk),4))


**中文**：观察结果，几个诚实的要点：
**English**: Observations, with honest takeaways:

**中文**：
1. **FunkSVD 把 RMSE 降到 ~0.97，ALS-WR ~1.01**，都碾压朴素 SVD（2.84！）和均值基线（~1.21），FunkSVD 还略好于上节邻域 CF（~1.0）——这就是 MF 在 Netflix Prize 大放异彩的原因。**诚实对比**：这里 ALS 略逊于 FunkSVD，是因为为了保持闭式解的简洁，ALS 版本只用全局均值去偏、没有加 per-user/item 偏置项；而 FunkSVD 带了偏置。这恰好印证了下面"偏置项不要省"的提醒——加上偏置 ALS 也能追平。
2. **隐因子数 k 有甜点**：RMSE 随 k 先降后趋平/略升，太大不仅慢还会过拟合（要靠 $\lambda$ 压制）。
3. **隐空间确实有语义**：动画、恐怖、纪录片等不同类型的电影在 2D 投影里出现了可见的分离倾向——而模型从未见过类型标签。但要诚实：分离**不是完美的**（只用评分、k 又小，且很多电影多类型混合），所以你会看到明显的重叠。这恰好说明隐因子捕捉的是"被谁喜欢"的协同信号，与人类的类型标签**相关但不等同**。

**English**:
1. **FunkSVD reaches RMSE ~0.97, ALS-WR ~1.01**, both crushing naive SVD (2.84!) and the mean baseline (~1.21); FunkSVD also edges out neighborhood CF (~1.0) — why MF shone in the Netflix Prize. **Honest comparison**: ALS here trails FunkSVD slightly because, to keep the closed form clean, the ALS version de-biases with the global mean only and adds *no* per-user/item bias terms, whereas FunkSVD carries biases. This very gap demonstrates the "don't drop biases" note below — add them and ALS catches up.
2. **#factors $k$ has a sweet spot**: RMSE falls then flattens/rises; too large is slow and overfits (held in check by $\lambda$).
3. **The latent space carries semantics**: Animation, Horror, Documentary, etc. show visible separation in the 2D projection — though the model never saw genre labels. But honestly: the separation is **not perfect** (only ratings, small $k$, many multi-genre films), so you will see real overlap. That itself is the point — latent factors capture "who-likes-it" collaborative signal, **correlated with but not identical to** human genre tags.

> 💼 **实战视角 / Practical angle**
> **中文**：MF = "为每个 ID 学一个 embedding"，这就是现代推荐的范式雏形。**SGD vs ALS**：SGD 省内存、适合在线增量；ALS 可并行、适合分布式与隐式反馈。**偏置项不要省**——光是 $\mu+b_u+b_i$（无隐向量）就能解释相当一部分评分方差，是非常强的基线。下一节我们处理工业界真正的数据形态——**隐式反馈**（点击/购买，只有正样本，没有评分）。
> **English**: MF = "learn an embedding per ID" — the prototype of modern recommendation. **SGD vs ALS**: SGD is memory-light and good for online/incremental updates; ALS is parallelizable, ideal for distributed and implicit feedback. **Do not drop the bias terms** — $\mu+b_u+b_i$ alone (no latent vectors) explains a big chunk of rating variance and is a strong baseline. Next we tackle industry's real data shape — **implicit feedback** (clicks/purchases: positives only, no ratings).

---
### 小结 / Summary
- **中文**：MF 把用户/物品嵌入低维隐空间，点积打分；只在观测评分上训练（FunkSVD/ALS），别用填0的朴素SVD。
- **English**: MF embeds users/items into a low-dim latent space, scores by dot product; train on observed ratings only (FunkSVD/ALS), not zero-filled naive SVD.
- **中文**：偏置 + 正则是标配；k 有甜点；SGD 适合在线、ALS 适合并行/隐式。
- **English**: Biases + regularization are standard; $k$ has a sweet spot; SGD suits online, ALS suits parallel/implicit.
- **中文**：隐空间自动学出类似类型的语义结构——这是"从行为学表示"的第一课。
- **English**: The latent space auto-discovers genre-like semantic structure — the first lesson of "learning representations from behavior."
